# GP10 — Clip / Intersection / Overlay

Περιορισμός σε περιοχή μελέτης και intersection δύο polygon layers.

In [ ]:
import geopandas as gpd
from pathlib import Path
root = Path('data/gp10')
study = gpd.read_file(root / 'study_area.geojson')
landuse = gpd.read_file(root / 'landuse.geojson')
protected = gpd.read_file(root / 'protected_zones.geojson')


In [ ]:
target_crs = 'EPSG:2100'
study = study.to_crs(target_crs)
landuse = landuse.to_crs(target_crs)
protected = protected.to_crs(target_crs)


In [ ]:
clipped = gpd.clip(landuse, study)
overlay = gpd.overlay(clipped, protected, how='intersection')
clipped
overlay


Έλεγξε CRS, rows, geometry types, validity, empty geometries, bounds και area πριν από export.

In [ ]:
clipped['area_m2'] = clipped.geometry.area
overlay['area_m2'] = overlay.geometry.area
for frame in (study, clipped, overlay):
    assert frame.crs.to_epsg() == 2100
    assert frame.geometry.notna().all()
    assert frame.geometry.is_empty.sum() == 0
    assert frame.geometry.is_valid.all()
assert (clipped['area_m2'] > 0).all()
assert (overlay['area_m2'] > 0).all()
print('areas:', clipped['area_m2'].sum(), overlay['area_m2'].sum())


In [ ]:
clipped.to_file('gp10_results.gpkg', layer='clipped_landuse', driver='GPKG')
overlay.to_file('gp10_results.gpkg', layer='landuse_protected', driver='GPKG')
print('export complete')
